In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", os.getenv("OPENAI_API_KEY"))  # Should print key
print("LANGCHAIN_API_KEY:", os.getenv("LANGCHAIN_API_KEY"))
print("LANGCHAIN_PROJECT:", os.getenv("LANGCHAIN_PROJECT"))

OPENAI_API_KEY: None
LANGCHAIN_API_KEY: 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
LANGCHAIN_PROJECT: GenAIAPPWithOPENAI


In [4]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
loader=WebBaseLoader("https://aws.amazon.com/what-is/generative-ai/")
loader

In [6]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Explained - AWS', 'description': 'Learn why generative AI is essential. Discover its benefits and how you can use it to create new content and ideas including text, conversations, images, video, and audio.', 'language': 'en-US'}, page_content="\n\n\n\n\n\n\n\n\n\n\n\nWhat is Generative AI? - Gen AI Explained - AWS\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to main content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nEnglish\n\n\n\nContact us\nSupport \n                   \n\n\n\nMy account \n                   \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nFilter: All\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSign in to console\nCreate account\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\

In [7]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [9]:
documents

[Document(metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Explained - AWS', 'description': 'Learn why generative AI is essential. Discover its benefits and how you can use it to create new content and ideas including text, conversations, images, video, and audio.', 'language': 'en-US'}, page_content='What is Generative AI? - Gen AI Explained - AWS\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSkip to main content\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nEnglish\n\n\n\nContact us\nSupport \n                   \n\n\n\nMy account \n                   \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nFilter: All\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSign in to console\nCreate account\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\

In [10]:
from langchain_community.embeddings import OllamaEmbeddings
embeddings = OllamaEmbeddings(model="gemma2:2b")


/var/folders/hr/mh6kzvp12034kdd80njjpk9m0000gn/T/ipykernel_74765/737296703.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="gemma2:2b")


In [11]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [12]:
vectorstoredb

In [15]:
## Query From a vector db
query="LangSmith has two usage limits: total traces and extended"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Extract and summarize data from any source for knowledge search functions.\nEvaluate and optimize different scenarios for cost reduction in areas like marketing, advertising, finance, and logistics.\nGenerate synthetic data to create labeled data for supervised learning and other ML processes.\n\n\n\n\n\n\nBoosts employee productivity'

In [14]:
from langchain_community.chat_models import ChatOllama
from langchain.schema import HumanMessage

llm = ChatOllama(model="gemma2:2b")
response = llm.invoke([HumanMessage(content="What is LangChain?")])
print(response.content)


/var/folders/hr/mh6kzvp12034kdd80njjpk9m0000gn/T/ipykernel_74765/2264754231.py:4: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gemma2:2b")


LangChain is a powerful open-source framework designed to simplify the process of building applications with large language models (LLMs). 

Think of it like this: LLMs are incredibly intelligent and can generate human-like text, translate languages, write code, summarize information, and much more. However, they lack the ability to interact with external data or perform specific tasks without explicit instructions. LangChain helps bridge that gap by providing a structured framework for:

**What LangChain Does:**

* **Chains together LLMs:** It allows you to combine different LLM calls (or even use multiple LLM outputs) in sequence to accomplish complex tasks, such as question answering, summarization, or code generation. This is like chaining together actions for a robot, making it capable of performing multi-step tasks. 
* **Connects with external data:**  It integrates with various data sources (APIs, databases, files), enabling LLMs to access and utilize real-world information to e

In [16]:
## Retrieval Chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
Answer the following question based only on the provided context:
<context>
{context}
</context>


"""
)

document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
| ChatOllama(model='gemma2:2b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [17]:
from langchain_core.documents import Document
document_chain.invoke({
    "input":"LangSmith has two usage limits: total traces and extended",
    "context":[Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
})

'What are the two usage limits that LangSmith enforces?  \n'

In [23]:
vectorstoredb

In [18]:
retriever=vectorstoredb.as_retriever()
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [19]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x10a1eef10>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nAnswer the following question based only on the provided context:\n<context>\n{context}\n</context>\n\n\n'), additional_kwargs={})])
            | ChatOll

In [20]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"LangSmith has two usage limits: total traces and extended"})
response['answer']

"This document describes how Generative AI can be applied across various industries, emphasizing ethical considerations and technical aspects. \n\nHere's a summary based on the provided text:\n\n**Key Applications:**\n\n* **Telecommunication:** Enhance customer service with human-like conversational agents, personalize sales assistance, optimize network performance through data analysis.\n* **Media & Entertainment:**  Produce novel content (animations, scripts, movies) at a fraction of traditional production costs and time.\n\n**Ethical Considerations:**\n\n* **Transparency:** Clearly communicate to users that they are interacting with AI using AI-specific identifiers or intro statements. This empowers user awareness and interaction choice.\n* **Security:** Implement robust security measures to prevent unauthorized access to sensitive data, utilizing masking techniques to protect PII. \n* **Data Bias:** Test extensively and address potential biases in the training data of Generative AI

In [21]:
response

{'input': 'LangSmith has two usage limits: total traces and extended',
 'context': [Document(id='54dc5e6d-95e5-4ddf-a32d-f1fb042a3dd5', metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Explained - AWS', 'description': 'Learn why generative AI is essential. Discover its benefits and how you can use it to create new content and ideas including text, conversations, images, video, and audio.', 'language': 'en-US'}, page_content='Extract and summarize data from any source for knowledge search functions.\nEvaluate and optimize different scenarios for cost reduction in areas like marketing, advertising, finance, and logistics.\nGenerate synthetic data to create labeled data for supervised learning and other ML processes.\n\n\n\n\n\n\nBoosts employee productivity'),
  Document(id='0226f466-7f2c-4129-aa94-e8741f3fb23f', metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Expl

In [28]:
response['context']

[Document(id='fea4ba4a-f48a-4cea-afbf-7f476db13475', metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Explained - AWS', 'description': 'Learn why generative AI is essential. Discover its benefits and how you can use it to create new content and ideas including text, conversations, images, video, and audio.', 'language': 'en-US'}, page_content='Extract and summarize data from any source for knowledge search functions.\nEvaluate and optimize different scenarios for cost reduction in areas like marketing, advertising, finance, and logistics.\nGenerate synthetic data to create labeled data for supervised learning and other ML processes.\n\n\n\n\n\n\nBoosts employee productivity'),
 Document(id='3b6ac5f6-ca70-4ef9-acf4-12e824bde2f3', metadata={'source': 'https://aws.amazon.com/what-is/generative-ai/', 'title': 'What is Generative AI? - Gen AI Explained - AWS', 'description': 'Learn why generative AI is essential. Discover its ben

In [22]:
print(ChatOllama(model="gemma2:2b").invoke([HumanMessage(content="Who are you ?")]).content)

I am Gemma, an AI assistant.  😊 

I'm a large language model created by the Gemma team. I can understand and generate human-like text. Ask me anything! 



In [23]:
print(ChatOllama(model="gemma2:2b").invoke([HumanMessage(content="Who are you ?")]).content)

I am Gemma, a large language model created by the Gemma team. I'm an open-weight AI assistant here to help you with text-based tasks! 😊  

Just keep in mind: 

* I'm a text-only model, so I can't interact with images or other media.
* I don't have access to real-time information or Google Search, so my knowledge is based on the data I was trained on. 


How can I help you today? 😄  

